## Creating fact table

In [0]:
%sql
CREATE OR REPLACE VIEW transport_lakehouse.gold.fact_private_vehicles AS
SELECT
  p.car_num,
  m.manufacturer_key,
  f.fuel_type_key,
  c.color_key,
  o.ownership_key,

  p.prod_year,
  p.road_entry_dt,
  p.last_test_dt,
  p.test_valid_until_dt,

  1 AS vehicle_count
FROM transport_lakehouse.silver.silver_vehicles_private p
LEFT JOIN transport_lakehouse.gold.dim_manufacturer m
  ON CAST(p.manufacturer_cd AS STRING) = CAST(m.manufacturer_cd AS STRING)
LEFT JOIN transport_lakehouse.gold.dim_fuel_type f
  ON p.fuel_type_nm = f.fuel_type_nm
LEFT JOIN transport_lakehouse.gold.dim_color c
  ON CAST(p.color_cd AS STRING) = CAST(c.color_cd AS STRING)
LEFT JOIN transport_lakehouse.gold.dim_ownership o
  ON p.ownership = o.ownership;


## Missing keys check

In [0]:
%sql
SELECT
  SUM(CASE WHEN manufacturer_key IS NULL THEN 1 ELSE 0 END) AS missing_manufacturer_key,
  SUM(CASE WHEN color_key IS NULL THEN 1 ELSE 0 END) AS missing_vehicle_type_key,
  SUM(CASE WHEN fuel_type_key IS NULL THEN 1 ELSE 0 END) AS missing_fuel_type_key,
  SUM(CASE WHEN ownership_key IS NULL THEN 1 ELSE 0 END) AS missing_ownership_key
FROM transport_lakehouse.gold.fact_private_vehicles;

## Duplication check

In [0]:
%sql
SELECT COUNT(*) AS rows, COUNT(DISTINCT car_num) AS distinct_cars
FROM transport_lakehouse.gold.fact_private_vehicles;


## Key null ratio check

In [0]:
%sql
SELECT
  CAST(AVG(CASE WHEN manufacturer_key IS NOT NULL THEN 1 ELSE 0 END) AS FLOAT) AS manufacturer_key_fill_rate,
  CAST(AVG(CASE WHEN ownership_key IS NOT NULL THEN 1 ELSE 0 END) AS FLOAT) AS ownership_key_fill_rate,
  CAST(AVG(CASE WHEN fuel_type_key IS NOT NULL THEN 1 ELSE 0 END) AS FLOAT) AS fuel_type_key_fill_rate,
  CAST(AVG(CASE WHEN color_key IS NOT NULL THEN 1 ELSE 0 END) AS FLOAT) AS color_key_fill_rate
FROM transport_lakehouse.gold.fact_private_vehicles;
